<a href="https://colab.research.google.com/github/karthika-k22/Text2Image-GAN/blob/main/T2I_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. INSTALL & AUTHENTICATION
!pip install huggingface_hub ipywidgets Pillow -q
import io, base64, torch
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output
from google.colab import userdata
from huggingface_hub import InferenceClient

# Get token (Case-insensitive check)
hf_token = next((userdata.get(k) for k in ['HF_TOKEN', 'HF_Token'] if userdata.get(k)), None)
client = InferenceClient(token=hf_token)

# 2. CUSTOM CSS STYLING (The "Attractive" Part)
style_css = """
<style>
    .main-box { border: 1px solid #444; border-radius: 15px; padding: 25px; background-color: #1e1e1e; box-shadow: 0px 10px 20px rgba(0,0,0,0.5); }
    .header-text { color: #00d4ff; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; text-align: center; }
    .gen-btn { font-weight: bold; background-color: #00d4ff !important; border-radius: 8px !important; }
    .prompt-box input { border-radius: 8px !important; border: 1px solid #555 !important; background: #2d2d2d !important; color: white !important; }
</style>
"""

# 3. UI COMPONENTS
header = widgets.HTML(f"{style_css}<div class='header-text'><h1>✨ Text 2 Image</h1></div>")

text_input = widgets.Text(
    value='A majestic tiger made of neon lights, cinematic 8k',
    placeholder='Enter a descriptive prompt...',
    layout=widgets.Layout(width='100%', margin='10px 0px')
)
text_input.add_class("prompt-box")

gen_button = widgets.Button(
    description='GENERATE ART',
    button_style='info', # Cyan/Blue
    layout=widgets.Layout(width='100%', height='45px')
)
gen_button.add_class("gen-btn")

download_container = widgets.Output(layout=widgets.Layout(margin='15px 0px'))
output_display = widgets.Output(layout=widgets.Layout(border='2px dashed #444', margin='10px 0px', padding='10px'))

# 4. LOGIC
def create_download_btn(img):
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    html = f'''
    <a href="data:image/png;base64,{img_str}" download="ai_masterpiece.png" style="text-decoration:none;">
        <div style="background:#00d4ff; color:black; padding:12px; text-align:center; border-radius:8px; font-weight:bold; cursor:pointer;">
            📥 SAVE TO DEVICE
        </div>
    </a>'''
    return widgets.HTML(html)

def on_click(b):
    with output_display:
        clear_output()
        download_container.clear_output()
        if not hf_token:
            print("❌ Authentication Error: Set HF_TOKEN in Secrets tab.")
            return

        print("AI is interpreting your prompt... Please wait.")
        try:
            # Using FLUX.1-schnell for professional quality
            image = client.text_to_image(text_input.value, model="black-forest-labs/FLUX.1-schnell")
            display(image.resize((600, 600)))
            with download_container:
                display(create_download_btn(image))
        except Exception as e:
            print(f"❌ API Error: {e}")

gen_button.on_click(on_click)

# 5. FINAL ASSEMBLY
dashboard = widgets.VBox([
    header,
    text_input,
    gen_button,
    download_container,
    output_display
], layout=widgets.Layout(width='600px', margin='0 auto'))

dashboard.add_class("main-box")
display(dashboard)

In [ ]:
# 1. INSTALL GRADIO
!pip install gradio huggingface_hub -q

import gradio as gr
from huggingface_hub import InferenceClient
from google.colab import userdata

# 2. AUTHENTICATION
hf_token = next((userdata.get(k) for k in ['HF_TOKEN', 'HF_Token'] if userdata.get(k)), None)
client = InferenceClient(token=hf_token)

# 3. GENERATION FUNCTION
def generate_art(prompt):
    if not hf_token:
        return None, "❌ Error: HF_TOKEN not found in Colab Secrets!"

    try:
        # Generate the image using FLUX.1
        image = client.text_to_image(prompt, model="black-forest-labs/FLUX.1-schnell")
        return image, "✨ Success! Art generated."
    except Exception as e:
        return None, f"❌ API Error: {e}"

# 4. DESIGN THE ATTRACTIVE UI (Gradio Blocks)
with gr.Blocks(theme=gr.themes.Soft(primary_hue="cyan")) as demo:
    gr.Markdown("# Image Creative Studio")
    gr.Markdown("Type a description below to generate high-fidelity AI art.")

    with gr.Row():
        with gr.Column(scale=1):
            prompt_input = gr.Textbox(
                label="Your Prompt",
                placeholder="e.g. A futuristic city in the style of cyberpunk...",
                lines=3
            )
            gen_btn = gr.Button("🚀 GENERATE ART", variant="primary")
            status_msg = gr.Markdown()

        with gr.Column(scale=1):
            image_output = gr.Image(label="Generated Result", type="pil")

    # Connect the button to the function
    gen_btn.click(
        fn=generate_art,
        inputs=prompt_input,
        outputs=[image_output, status_msg]
    )

# 5. LAUNCH WITH PUBLIC URL
# share=True creates the separate URL you can send to others
demo.launch(share=True, debug=True)